# Cloud Security Intelligence Platform
Intermediate SQL

Scenario: You're a data scientist supporting a cloud security team.

Security leadership wants to identify which identities are experiencing the highest concentration of failed events.

Using the events and identities tables from Days 17–18, create a report that returns the two highest-failure-rate identities within each department.

### Your report must contain:
Column	Description
department	Identity's department
user	CloudTrail identity
total_events	Total number of events
failed_events	Number of failed events
failure_rate	Percentage of events that failed
department_rank	Rank within that department by failure rate

### Requirements:
- Include identities with zero failures.
- Rank users independently within each department.
- Break equal failure rates by giving the higher rank to the user with more total events; if both tie, use alphabetical username order.
- Return no more than two identities per department.
- Sort results by department and department rank.
- Use the clean identities table, not identities_duplicate.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "cloudtrail.db"

assert db_path.exists(), "SQLite database not found"

connection = sqlite3.connect(db_path)

In [2]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_events FROM events",
    connection
)

,total_events
0,95


In [5]:
pd.read_sql_query(
    "SELECT * FROM identities",
    connection
)

,user,department,role
0,admin,IT,Administrator
1,developer,Engineering,Data Engineer
2,analyst,Finance,Data Analyst
3,service-user,IT,Service Account


In [7]:
pd.read_sql_query(
    "SELECT * FROM events where failed=1",
    connection
)

,timestamp,user,action,region,failed
0,2026-11-11T07:00:00Z,admin,ConsoleLogin,eu-west-1,1
1,2026-11-11T04:00:00Z,service-user,PutObject,us-west-2,1
2,2026-11-11T21:00:00Z,admin,ConsoleLogin,us-east-1,1
3,2026-11-12T17:00:00Z,service-user,PutObject,us-west-2,1
4,2026-11-14T05:00:00Z,admin,PutObject,eu-west-1,1
5,2026-11-14T01:00:00Z,developer,ConsoleLogin,us-east-1,1
6,2026-11-14T22:00:00Z,developer,ConsoleLogin,eu-west-1,1
7,2026-11-10T11:00:00Z,admin,PutObject,us-east-1,1
8,2026-11-12T05:00:00Z,service-user,PutObject,eu-west-1,1


In [15]:
pd.read_sql_query(
    """
    
        SELECT
            user,
            COUNT(*) AS total_events,
            SUM(failed) AS failed_events,
            (SUM(failed) * 1.0 / COUNT(*) * 100) AS failure_rate
        FROM events
        GROUP BY user

    """,
    connection
)

,user,total_events,failed_events,failure_rate
0,admin,36,4,11.111111
1,analyst,18,0,0.000000
2,developer,22,2,9.090909
3,service-user,19,3,15.789474


In [ ]:
# first attempt
pd.read_sql_query(
    """
   SELECT 
        i.department,
        i.user,
        COUNT(*) AS total_events,
        SUM(e.failed) as failed_events,
        (SUM(e.failed) * 1.0 / COUNT(*) * 100) AS failure_rate,
        ROW_NUMBER() OVER (
            PARTITION BY department) as department_rank
    FROM events as e
    LEFT JOIN identities as i
        ON i.user=e.user
    GROUP BY i.department, i.user
    
    """,
    connection
)

,department,user,total_events,failed_events,failure_rate,department_rank
0,Engineering,developer,22,2,9.090909,1
1,Finance,analyst,18,0,0.000000,1
2,IT,admin,36,4,11.111111,1
3,IT,service-user,19,3,15.789474,2


In [30]:
# second attempt
ranked_users = pd.read_sql_query(
    """
WITH user_activity AS (
    SELECT
        i.department,
        e.user,
        COUNT(*) AS total_events,
        SUM(e.failed) AS failed_events,
        SUM(e.failed) * 1.0 / COUNT(*) * 100 AS failure_rate
    FROM events AS e
    LEFT JOIN identities AS i
        ON e.user = i.user
    GROUP BY i.department, e.user
),
ranked_users AS (
    SELECT
        department,
        user,
        total_events,
        failed_events,
        failure_rate,
        ROW_NUMBER() OVER (
            PARTITION BY department
            ORDER BY failure_rate DESC, total_events DESC, user
    ASC ) AS department_rank
    FROM user_activity
)
SELECT *
FROM ranked_users
WHERE department_rank <= 2
ORDER BY department, department_rank
""", connection
)

In [31]:
print("Number of rows in users:", len(ranked_users))
print("How many unique values: ", ranked_users["user"].nunique())

assert len(ranked_users) == ranked_users["user"].nunique()
assert ranked_users["user"].isnull().sum() == 0

Number of rows in users: 4
How many unique values:  4


In [33]:
assert ranked_users.groupby("department").size().max() <= 2

In [36]:
df_identities = pd.read_sql_query(
    "SELECT * FROM identities",
    connection
)

assert df_identities["user"].duplicated().sum() == 0

### Day 21 — SQL Interview Practice

Completed an independent SQL challenge using the Cloud Security Intelligence Platform's synthetic CloudTrail dataset.

**Objective:** Identify the two highest-failure-rate identities within each department.

**Key accomplishments:**
- Joined CloudTrail events with identity metadata.
- Calculated total events, failed events, and failure rates by identity.
- Used CTEs to separate aggregation, ranking, and filtering.
- Applied `ROW_NUMBER()` with `PARTITION BY department`.
- Implemented tie-breaking using failure rate, total events, and username.
- Filtered results to a maximum of two identities per department.
- Validated department limits and unique identity keys.

**Key lessons:**
- `PARTITION BY` determines where rankings restart.
- `ORDER BY` inside a window function determines ranking priority.
- Window-function results can be filtered in an outer query.
- Source-level validation is different from validating a filtered analytical report.

**Interview reflection:** My initial query calculated the required aggregates but didn't fully implement the ranking rules. I improved the solution by introducing CTEs, adding deterministic ranking criteria, and filtering the final results.

**Career action:** [Record the application or networking action completed, including the employer, role, date, and outcome.]